In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import skimage.io
import os 
import tqdm
import glob
import tensorflow 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import tensorflow as tf
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.data.experimental import AUTOTUNE
from tensorflow.keras import Sequential, Input, Model
from tensorflow.keras.layers import RandomRotation, RandomZoom
from tensorflow.keras.layers.experimental.preprocessing import Rescaling
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras import applications
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD

from tqdm import tqdm
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from skimage.io import imread, imshow
from skimage.transform import resize

from tensorflow.keras.models import Sequential
from tensorflow.keras.metrics import Precision, AUC,Recall
from tensorflow.keras.layers import InputLayer, BatchNormalization, Dropout, Flatten, Dense, Activation, MaxPool2D, Conv2D
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

import copy
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
#import cv2
import keras
from keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import load_img, img_to_array
import matplotlib
import matplotlib.pylab as plt
#import seaborn as sns
from sklearn.utils import shuffle
from sklearn.metrics import confusion_matrix
from keras.applications.vgg16 import VGG16,preprocess_input
from keras.applications.vgg19 import VGG19,preprocess_input
from tensorflow.keras.utils import image_dataset_from_directory
#import tensorflow_addons as tfa
#from tensorflow_addons.optimizers import RectifiedAdam
#from tensorflow_addons.optimizers import AdamW

labels = pd.read_csv('AlzheimersFLAIR/patient_labels.csv')
labels

In [ ]:
example_list = [[0, 1, 2, 3, 4, 5, 6, 7],
                [8, 9, 10, 11, 12, 13, 14, 15],
               [16, 17, 18, 19, 20, 21, 22, 23]]



In [ ]:
from keras.preprocessing.image import ImageDataGenerator

BATCH_SIZE=32

SEED=1345

train_datagen=ImageDataGenerator(rescale=1./255,
                                shear_range=0,
                                zoom_range=0.2, 
                                rotation_range = 30, height_shift_range = 0.02, 
                                width_shift_range = 0.02
                                )

validation_datagen=ImageDataGenerator(rescale=1./255)
test_datagen=ImageDataGenerator(rescale=1./255)


#Defining directories for train,validation,test 
train_dir = 'Splitted2D-largest2/train'
validation_dir = 'Splitted2D-largest2/val'
test_dir = 'Splitted2D-largest2/test'


#Defining generatores for train,validation,test

train_generator=train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    shuffle=True,
    seed = SEED,
    batch_size= 32,
    class_mode ='categorical',
)

validation_generator = validation_datagen.flow_from_directory(
    validation_dir,
    target_size=(224, 224),
    seed = SEED,
    shuffle=True,
    batch_size= 32,
    class_mode ='categorical',
)

# Define generator for test set using flow_from_directory
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    shuffle=True,
    seed = SEED,
    batch_size = 32,
    class_mode ='categorical',
)

In [ ]:
input_folder='jpg-slice19'

class_count=dict()
class_names=list(train_generator.class_indices.keys())
print(class_names)

for i in class_names:
    name = 'CN'
    if (i == '1'): 
        name = 'EMCI'
    elif (i == '2'):
        name = 'LMCI'
    elif (i == '3'):
        name = 'AD'
    print(name, i)
    class_count[name]=len(os.listdir(input_folder+'/'+i))

plt.figure(figsize=(10,4))
colors = ['red', 'blue', 'green', 'orange']
plt.bar(class_count.keys(),class_count.values(),color=colors)

plt.xlabel('Classes')
plt.ylabel('Counts')
plt.title('Visualization of Data Imbalance')
plt.grid(True)
plt.show()

total_samples=sum(class_count.values())

for i in range(4):
    class_weight = round(total_samples / (4 * list(class_count.values())[i]), 2)
    print(f'Weight for class \"{class_names[i]}\" : {class_weight}')

In [ ]:
filepath = 'vgg16-results/VGG16-largest2V1/bestVal/vgg16-largest2V1-bestVal.h5'
earlystopping=EarlyStopping(monitor='val_accuracy',
                           mode='max',
                           patience=35,
                           verbose=1)

checkpoint=ModelCheckpoint(filepath,monitor = 'val_accuracy', 
                                mode='max', 
                                save_best_only=True, 
                                verbose = 1)

callback_list=[earlystopping,checkpoint]

In [ ]:
print("test")

In [ ]:
from keras.backend import sigmoid
def swish(x, beta = 1.3):
    return (x * sigmoid(beta * x))

from keras.utils import get_custom_objects
from keras.layers import Activation
get_custom_objects().update({'swish': Activation(swish)})

In [ ]:
from tensorflow.keras.optimizers.legacy import SGD, Adam, Nadam

input_shape = (224,224, 3)

#Create an instance of the VGG16 model
vgg16 = VGG16(include_top=False, input_shape=input_shape,
                   weights='imagenet')

# Freeze the layers of the VGG16 model
for layer in vgg16.layers:
    layer.trainable = False
    
# Create a new model with additional layers

#tf.keras.layers.LeakyReLU(0.1)
# previously this was global_average_pooling
global_average_layer = tf.keras.layers.GlobalAveragePooling2D()
#global_max_layer = tf.keras.layers.GlobalMaxPooling2D()

#init = tf.keras.initializers.HeNormal(seed=42)
prediction_layer = keras.layers.Dense(4,activation='softmax')

model = tf.keras.Sequential([vgg16, global_average_layer,
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(2048, activation = 'swish', kernel_initializer = 'glorot_normal'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(1024, activation='swish', kernel_initializer = 'glorot_normal'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(512, activation='swish', kernel_initializer = 'glorot_normal'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(256, activation='swish', kernel_initializer = 'glorot_normal'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(128, activation='swish', kernel_initializer = 'glorot_normal'),
    keras.layers.Dropout(0.3),
    prediction_layer
])

model.compile(optimizer=SGD(learning_rate = 0.005, decay = 0.0012), 
              loss='categorical_crossentropy', metrics=['accuracy',tf.keras.metrics.AUC(),
                        tf.keras.metrics.Precision(),
                        tf.keras.metrics.Recall(),])

In [ ]:
vgg16.summary()

In [ ]:
print("test")

In [ ]:

history=model.fit(train_generator,
                        validation_data=validation_generator,
                        steps_per_epoch=len(train_generator),
                        epochs = 100,
                        verbose = 1,
                 callbacks = callback_list)

In [ ]:
model.save('vgg16-results/VGG16-largest2V1/stopped/vgg16-largest2V1-stop.h5')

In [ ]:
from classification_models_3D.tfkeras import Classifiers
from keras.layers import Conv3D, MaxPooling3D, BatchNormalization, Dropout, Dense, Flatten
from keras.optimizers.legacy import SGD, Adam

VGG16, preprocess_input = Classifiers.get('vgg16')
conv_layers = VGG16(input_shape=(40, 64, 64, 3), weights='imagenet')

type(conv_layers.layers)

model_input=tf.keras.Input(shape=(20, 64, 64, 3))
index = 0
x=model_input
for layer in conv_layers.layers[1:]:
    if isinstance(layer, MaxPooling3D):
        print(index, layer)
        kwargs=layer.get_config()
        x=MaxPooling3D(pool_size=(2, 2, 2), strides=(1, 2, 2))(x)
        #x = Custom3DBatchNormalization()(x)
    else:
        print(index, layer)
        x=layer(x)
    index += 1
conv_layers=tf.keras.Model(inputs=model_input, outputs=x, name="vgg16")
conv_layers.summary()

conv_layers.summary()

# for layer in conv_layers.layers:
#     layer.trainable = False

from keras.layers import Conv3D, MaxPooling3D, BatchNormalization, Dropout, Dense, Flatten
from keras.optimizers.legacy import SGD, Adam
# use heNormal initialization



global_average_layer = tf.keras.layers.GlobalAveragePooling3D(name = 'global_average')

prediction_layer = keras.layers.Dense(3,activation='softmax', name = 'prediction_layer', kernel_initializer = 'glorot_normal')
#flatten = Flatten()

# big overfitting problem (val_loss hasn't changed at all) --> put in more dropouts, put decay in optimizer
# try getting rid of the custom batchnorms in the conv_layers because batchnorm wasn't supposed to be there in 
# the first place

# try adding more dense layers

# you might need to unfreeze the conv layers because the conv layers are just a 3d expansion of vgg16 2D
# change up the maxpooling3D so not all of them are just (1, 2, 2) so there is a bit of 3d summarization
dense_layers = Sequential([
    BatchNormalization(),
    keras.layers.GaussianDropout(0.1),
    keras.layers.Dense(256, activation='relu', kernel_initializer = 'glorot_normal'),
    #keras.layers.Dense(1, activation='relu', kernel_initializer = 'glorot_normal'),
   # keras.layers.Dropout(0.1),
    #keras.layers.Dense(64, activation='swish', kernel_initializer = 'glorot_normal'),
    #keras.layers.Dropout(0.1),
], 
    name = 'dense_layers')

model = Sequential([
    conv_layers,
    global_average_layer, 
    #flatten,
    dense_layers, 
    prediction_layer
])

model.compile(optimizer=SGD(learning_rate = 0.005, decay = 0.000001),
              loss='categorical_crossentropy', metrics=['accuracy',tf.keras.metrics.AUC(),
                        tf.keras.metrics.Precision(),
                        tf.keras.metrics.Recall(),])

In [ ]:
from keras.utils import plot_model
plot_model(model, to_file='vgg16-3dV3.png', show_shapes=True, show_layer_names=True, rankdir = "LR")

In [ ]:
pip install graphviz

In [ ]:
model = keras.models.load_model('vgg16-results/VGG16-largest2V1/bestVal/vgg16-largest-pretrainedV3-bestVal.h5')

In [ ]:
model.load_weights('vgg16-results/VGG16-largest2V1/bestVal/vgg16-largest2V1-bestVal.h5')

In [ ]:
result = model.evaluate(train_generator)
train_loss = result[0]
train_accuracy = result[1]
train_AUC = result[2]
train_pre = result[3]
train_rec = result[4]
print(f'Train Loss = {train_loss}')
print(f'Train Accuracy = {train_accuracy}')
print(f'Train AUC = {train_AUC}')
print(f'Train Precision = {train_pre}')
print(f'Train Recall = {train_rec}')

result = model.evaluate(test_generator)
test_loss = result[0]
test_accuracy = result[1]
test_AUC = result[2]
test_pre = result[3]
test_rec = result[4]
print(f'Test Loss = {test_loss}')
print(f'Test Accuracy = {test_accuracy}')
print(f'Test AUC = {test_AUC}')
print(f'Test Precision = {test_pre}')
print(f'Test Recall = {test_rec}')

In [ ]:
y_pred=[]
y_true = []
count = 0
count2 = 0
for images, label in test_generator:
    for l in label:
        count2 += 1
        y_true.append(np.argmax(l))
        if (count2 >= 300):
            break
    
    Y_pred = [np.argmax(l) for l in model.predict(images)]
    will_break = False
    for l in Y_pred:
        count += 1
        y_pred.append(l)
        if (count >= 300):
            will_break = True
            break
    if (will_break):
        break
    

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_confusion_matrix(cm, classes, normalize=False, cmap=plt.cm.Blues):
    if normalize:
#         cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        title = "VGG16 Normalized Confusion Matrix"
    else:
        title = "VGG16 Confusion Matrix"

    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=90)
    plt.yticks(tick_marks, classes)

    fmt = 'd'
#     '.2f' if normalize else 
    thresh = cm.max() / 2.
    for i, j in np.ndindex(cm.shape):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()

cm = confusion_matrix(y_true, y_pred)

class_names = ['CN', 'EMCI', 'LMCI', 'AD']

plt.figure(figsize=(8, 6))
plot_confusion_matrix(cm, class_names, normalize=False)
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
# Compute the classification report
report = classification_report(y_true, y_pred)
print("VGG16 Classification Report:")
print(report)

In [ ]:
print(history.history.keys())
# summarize history for accuracy
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()
# summarize history for loss
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()